##### 09 - Pretrained Vision Model

##### Objective

Build a transfer-learning image classifier using pretrained ResNet18.

This notebook will:

1. Load pretrained ResNet18 weights.
2. Inspect the original ResNet18 architecture.
3. Understand the original 1000-class classifier.
4. Freeze the pretrained feature-extraction layers.
5. Replace the original classifier with a new 2-class classifier.
6. Identify frozen and trainable parameters.
7. Use preprocessing associated with the pretrained weights.
8. Perform a forward-pass smoke test.

No model training is performed in this notebook.


1. First: what is ResNet18?

ResNet stands for Residual Network. The 18 refers to the standard ResNet-18 architecture's 18 learnable layers counted under its naming convention.

Conceptually, it's much deeper than our small custom CNN:

``` text

Conv
 ↓
Conv
 ↓
Conv
 ↓
Global pooling
 ↓
Linear
 ↓
Linear


ResNet18

Conv
 ↓
Residual blocks
 ↓
Residual blocks
 ↓
Residual blocks
 ↓
Residual blocks
 ↓
Global pooling
 ↓
Fully connected classifier

```

The important new idea is the residual connection.

A simplified ordinary block might learn:

``` text

x
│
↓
layers
│
↓
F(x)

A residual block adds the original input back:

       ┌──────────────┐
       │              │
x ─────┤              ↓
       ↓              +
    layers            │
       ↓              ↓
      F(x) ───────→ F(x) + x
      

```
This is often called a skip connection or shortcut connection.

You don't need to understand the mathematics deeply yet. For this project, remember:

Residual connections help make deeper neural networks easier to train.

##### 2. Imports

In [0]:
import torch

from torchvision.models import (
    ResNet18_Weights,
    resnet18,
)

from src.data_preparation import (
    create_image_dataloaders,
)

from src.vision_model import (
    build_resnet18_classifier,
)

##### 3. Recreate Dataset Metadata

In [0]:
data = create_image_dataloaders()

print(
    "Class names:",
    data.class_names,
)

print(
    "Class mapping:",
    data.class_to_idx,
)

print(
    "Number of classes:",
    data.num_classes,
)

##### Dataset Metadata

The dataset is recreated independently so that the model can derive its
number of output classes from the actual dataset rather than hard-coding
the value.

The current DataLoaders use the preprocessing pipeline created for the
custom CNN.

In this notebook, the dataset object is used only for metadata such as:

- class names
- class-to-index mapping
- number of classes

Real dataset images are not passed through pretrained ResNet18 in this
notebook.

Pretrained-compatible image preprocessing will be introduced before
training the transfer-learning model.

##### 4. Select Pretrained ResNet18 Weights

In [0]:
weights = ResNet18_Weights.DEFAULT

print(
    "Selected pretrained weights:",
    weights,
)

##### Pretrained Weights

`resnet18` defines the ResNet18 neural-network architecture.

`ResNet18_Weights` identifies available pretrained parameter sets for
that architecture.

`ResNet18_Weights.DEFAULT` selects the recommended pretrained weight
configuration.

The selected weights were learned from the ImageNet-1K classification
task.

Pretrained weights provide previously learned visual representations;
they do not mean that ResNet18 already understands casting defects.


``` text

weights=None

ResNet18 architecture
        +
randomly initialized weights


weights=ResNet18_Weights.DEFAULT

ResNet18 architecture
        +
pretrained weights

```

So resnet18() defines the architecture, while weights= determines how its parameters are initialized.

##### 5. Load the Original Pretrained ResNet18

In [0]:
original_model = resnet18(
    weights=weights,
)

print(original_model)

The downloaded checkpoint is stored in the local PyTorch cache associated
with the Databricks compute environment.

It is not the project's Unity Catalog Volume and it is not the MLflow
model artifact store.

The cache allows PyTorch to reuse the downloaded pretrained weights while
they remain available on that compute environment.

##### 6. Inspect the Original Classifier



In [0]:
print(
    "Original classifier:"
)

print(
    original_model.fc
)

print(
    "Classifier input features:",
    original_model.fc.in_features,
)

print(
    "Original output classes:",
    original_model.fc.out_features,
)

Pretrained ResNet18
        ↓
512 learned visual features
        ↓
Linear(512 → 1000)
        ↓
1000 ImageNet logits

##### Original Classifier

ResNet18 converts each image into a learned visual representation
containing 512 features.

The original final fully connected layer receives those 512 features and
produces 1000 raw output scores, or logits, corresponding to the original
ImageNet-1K classes.

Each of the 1000 output neurons receives all 512 input features.

Therefore, the original classifier contains:

- 512 × 1000 = 512,000 weights
- 1000 biases

Total classifier parameters:

512,000 + 1,000 = 513,000

##### 7. Inspect the Pretrained Image Preprocessing

In [0]:
pretrained_transform = (
    weights.transforms()
)

print(
    pretrained_transform
)

##### Pretrained Image Preprocessing

The pretrained weights have an associated image preprocessing pipeline.

Conceptually:

Raw image
    ↓
Resize while preserving aspect ratio
    ↓
Center crop to 224 × 224
    ↓
Convert image to the representation expected by the model
    ↓
Normalize RGB channels
    ↓
Pretrained ResNet18

The resize size of 256 means that the image is resized according to the
pretrained transformation while preserving its aspect ratio.

For example, an image approximately:

    400 × 300

could first become approximately:

    341 × 256

rather than being forced directly into a square.

A 224 × 224 region is then obtained for model input.

The RGB normalization values associated with these pretrained weights are:

Mean:
    [0.485, 0.456, 0.406]

Standard deviation:
    [0.229, 0.224, 0.225]

These values should not be memorized or manually assumed for every vision
model. The preprocessing associated with the selected pretrained weights
should be used.

##### 8. Build the Project ResNet18 Classifier

In [0]:
model = build_resnet18_classifier(
    num_classes=data.num_classes,
    freeze_backbone=True,
)

print(model)

##### 9. Inspect the New Project Classifier

In [0]:
print(
    "Project classifier:"
)

print(
    model.fc
)

print(
    "Classifier input features:",
    model.fc.in_features,
)

print(
    "Project output classes:",
    model.fc.out_features,
)

##### Project Classifier

The pretrained ResNet18 backbone continues to produce a 512-feature
visual representation.

The original 1000-class ImageNet classifier is no longer appropriate
for the product-defect problem.

The project dataset contains two classes:

- `def_front`
- `ok_front`

Therefore, the original:

    Linear(512 → 1000)

is replaced by:

    Linear(512 → 2)

The two output neurons produce two raw logits, one for each project
class.

##### 10. Inspect Trainable Parameters

In [0]:
for name, parameter in model.named_parameters():

    if parameter.requires_grad:

        print(
            "TRAINABLE:",
            name,
            tuple(parameter.shape),
        )

fc.weight (2, 512)

2 output neurons × 512 weights per output neuron = 1,024 weights

fc.bias (2,)

1 bias per output neuron × 2 output neurons = 2 biases


1,024 weights +    2 biases = 1,026 trainable parameters

##### 11. Count Total, Frozen, and Trainable ParametersHow can frozen layers still be useful?

In [0]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

frozen_parameters = (
    total_parameters
    - trainable_parameters
)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

print(
    "Frozen parameters:",
    f"{frozen_parameters:,}",
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}",
)

##### Frozen vs Trainable Parameters

Freezing a layer does not remove it from the model.

Frozen pretrained layers still participate in the forward pass and
continue producing learned visual features.

Freezing means that their parameters are not updated during training.

The pretrained ResNet18 backbone is therefore used as a visual feature
extractor, while the new classification layer remains trainable.

Frozen does not mean: Don't use these layers.

It means: Use these layers during the forward pass, but don't update their learned parameters.


``` text

Casting image
      ↓
Pretrained Conv layers 🔒
      ↓
Pretrained residual blocks 🔒
      ↓
512 visual features
      ↓
New Linear layer 🔓
      ↓
2 logits

```

Every image still passes through the entire network.

The frozen part acts as our feature extractor.

##### 12. Verify Trainable Parameter Names

In [0]:
frozen_parameter_names = []
trainable_parameter_names = []

for name, parameter in model.named_parameters():

    if parameter.requires_grad:

        trainable_parameter_names.append(
            name
        )

    else:

        frozen_parameter_names.append(
            name
        )

print(
    "Number of frozen parameter tensors:",
    len(frozen_parameter_names),
)

print(
    "Trainable parameter tensors:",
    trainable_parameter_names,
)

##### 13. Create Dummy Image Batch

In [0]:
#We are not using real casting images yet.

#Instead, use a random tensor to test whether the architecture works correctly.

dummy_images = torch.randn(
    4,      # number of images / batch size
    3,      # RGB channels
    224,    # image height
    224,    # image width
)

print(
    "Dummy input shape:",
    dummy_images.shape,
)

##### 14. Forward-Pass Smoke Test

In [0]:
model.eval()

with torch.no_grad():

    logits = model(
        dummy_images
    )

print(
    "Output logits shape:",
    logits.shape,
)

4 images
×
2 raw class scores per image

##### 15. Inspect Raw Logits

In [0]:
print(
    logits
)

The outputs are raw logits.

They should not be interpreted as meaningful product-defect predictions
because:

1. the input images are random dummy tensors, and
2. the new two-class classification layer has not been trained.

The purpose of this step is only to verify that the model accepts the
expected image tensor shape and produces the expected output shape.

##### 16. Final Architecture Validation

In [0]:
assert (
    model.fc.in_features
    == 512
)

assert (
    model.fc.out_features
    == data.num_classes
)

assert (
    trainable_parameters
    == 1_026
)

assert (
    logits.shape
    == (4, data.num_classes)
)

assert (
    trainable_parameter_names
    == ["fc.weight", "fc.bias"]
)

print(
    "ResNet18 architecture validation passed."
)

##### 17. 🔍 Explainable AI Observation


ResNet18 transforms an input image through many convolutional and
residual layers before producing its final classification logits.

The final prediction alone does not explain which image regions
influenced the model's decision.

Later, Grad-CAM will use information from the convolutional feature maps
and model gradients to investigate which regions influenced selected
predictions.

Grad-CAM identifies regions that influenced the model's prediction. It
does not, by itself, establish the ground-truth physical location of a
defect.

##### 18. ⚖️ Responsible AI Observation


The pretrained ResNet18 backbone learned its visual representations from
a different source dataset and classification task.

Transfer learning allows those visual representations to be reused, but
the pretrained model should not be assumed to already understand casting
defects.

The adapted model must therefore be evaluated using the casting-defect
dataset.

Evaluation should include not only overall accuracy but also
class-specific performance, missed defects, false defect alarms, and
failure cases.

Pretrained models may also carry limitations associated with their
original training data, making downstream validation important.

##### 19. Key Learnings


1. `resnet18` defines the ResNet18 neural-network architecture.

2. `ResNet18_Weights` identifies pretrained weight configurations
   available for ResNet18.

3. `ResNet18_Weights.DEFAULT` selects the recommended pretrained
   ResNet18 weights.

4. The pretrained ResNet18 backbone produces a 512-feature visual
   representation for each image before the final classifier.

5. The original classifier is `Linear(512, 1000)` because the original
   ImageNet-1K classification task contains 1000 output classes.

6. Each output neuron has one weight for each input feature and one
   bias.

7. Our dataset contains two classes: `def_front` and `ok_front`.

8. The original classifier is therefore replaced with
   `Linear(512, 2)`.

9. `Linear(512, 2)` contains 1,024 weights and 2 biases, giving 1,026
   trainable parameters.

10. The pretrained backbone is frozen during feature extraction, while
    the new classification layer remains trainable.

11. Frozen layers still participate in the forward pass; their
    parameters simply are not updated during training.

12. Pretrained weights have an associated image preprocessing pipeline.

13. A PyTorch image batch uses `[B, C, H, W]`, where `B` represents the
    batch size.

14. A dummy input with shape `[4, 3, 224, 224]` produces two logits for
    each image, giving an output shape of `[4, 2]`.

15. A dummy forward pass validates the architecture but does not measure
    classification performance.

16. Transfer learning reuses previously learned visual representations;
    it does not mean that ResNet18 already understands casting defects.

##### 20. Notebook Conclusion


A pretrained ResNet18 model was successfully adapted to the product
defect classification architecture.

The dataset contains two classes:

- `def_front`
- `ok_front`

The number of output classes was derived dynamically from the dataset
metadata using `data.num_classes`.

The original ImageNet classifier:

    Linear(512 → 1000)

was replaced with:

    Linear(512 → 2)

The pretrained ResNet18 backbone was frozen, leaving only 1,026
parameters in the new classification layer trainable.

The preprocessing associated with the pretrained ResNet18 weights was
also inspected and distinguished from the preprocessing previously used
for the custom CNN.

A forward-pass smoke test confirmed that a dummy batch with shape:

    [4, 3, 224, 224]

produces output logits with shape:

    [4, 2]

The transfer-learning model architecture is therefore ready.

Before real casting images are used for transfer-learning training, the
data pipeline must be updated to use preprocessing compatible with the
pretrained ResNet18 weights.